# 15장 데이터셋과 DataLoader — 내 데이터를 배치로

**3단계 · 다시 만들기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_3_build.ipynb)

이 노트북에는 **주석만 남긴 코드**가 들어 있습니다. 책과 1단계 노트북을 덮고, 주석을 실마리 삼아 코드를 되살려 보세요. 아래 표의 「다시 만들기」 과제가 이 노트북의 목표입니다.

- 코드 셀 첫 줄의 `%%writefile 파일이름` 은 "이 셀의 내용을 그 파일로 저장하라"는 뜻입니다. 채운 뒤 셀을 실행해 저장하고, 확인 셀로 돌려 봅니다.
- 막히면 오류 메시지를 먼저 읽고(부록 A), 그래도 안 되면 1단계 노트북에서 그 부분만 확인하세요.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `make_shapes.py` 로 데이터를 만들고 `p01`~`p03` 을 치고 실행 | [ch15_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_1_follow.ipynb) |
| 2. 주석 달기 | `ImageFolderDataset` 의 `__init__` 과 `__getitem__` 에 "언제 실행되는가(한 번 / 매 항목)"를 주석으로 | [ch15_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `ImageFolderDataset` 복원. 그다음 14장 패키지의 `data/` 에 `shapes.py` 로 넣고 `--dataset shapes` 로 학습되게 등록 | [ch15_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_3_build.ipynb) **(지금 이 노트북)** |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "15"))                      # 이 장의 코드 폴더로 이동
if not os.path.isdir(os.path.join(os.environ["JUCOS_DATA"], "shapes")):   # 15장의 도형 데이터가 없으면 만든다
    !python ../15/make_shapes.py
print("준비 끝. 작업 폴더: code/15")

## `p02_custom_dataset.py` — 대상: `ImageFolderDataset`

In [ ]:
%%writefile p02_custom_dataset.py
# 15.3~15.4 내 이미지 폴더를 Dataset 으로 만들고, transform 으로 전처리·증강한다
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")


class ImageFolderDataset(Dataset):
    """폴더 이름이 정답인 이미지 폴더. root/{클래스}/{파일}.png"""

    def __init__(self, root, transform=None):
        # ['circle', 'square', 'triangle']
        # 이름 → 번호
        # (경로, 번호) 목록
        raise NotImplementedError("여기를 채우세요")

    def __len__(self):
        raise NotImplementedError("여기를 채우세요")

    def __getitem__(self, i):
        # 파일 → PIL 이미지 (여기서 비로소 읽는다)
        # PIL → 텐서 (+ 전처리)
        raise NotImplementedError("여기를 채우세요")


if __name__ == "__main__":          # import 될 때는 아래 실험이 돌지 않게 (4장)
    # ---- transform 없이: 목록만 만든다. 이미지는 아직 안 읽는다
    ds = ImageFolderDataset(os.path.join(ROOT, "train"))
    print("클래스:", ds.classes, ds.class_to_idx)
    print("len:", len(ds), "| items[0] =", ds.items[0][0].replace(ROOT, "…"), ds.items[0][1])
    img, label = ds[0]
    print("ds[0] ->", type(img).__name__, img.size, img.mode, "| label", label)

    # ---- 기본 transform: 텐서로 바꾸고 표준화
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
    basic = transforms.Compose([
        transforms.Resize((32, 32)),           # 크기 통일
        transforms.ToTensor(),                 # [3, 32, 32], 0~1
        transforms.Normalize(mean, std),       # (x - mean) / std → 대략 -1~1
    ])
    ds = ImageFolderDataset(os.path.join(ROOT, "train"), transform=basic)
    img, label = ds[0]
    print("transform 후:", img.shape, img.dtype, f"범위 {img.min():.2f}~{img.max():.2f}", "| label", label)

    # ---- 증강(augmentation): 학습 때만, 매번 다르게
    augment = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    ds_aug = ImageFolderDataset(os.path.join(ROOT, "train"), transform=augment)
    torch.manual_seed(0)
    a, _ = ds_aug[0]
    b, _ = ds_aug[0]
    print("같은 이미지를 두 번 꺼내면 다른가:", not torch.equal(a, b))

    # ---- DataLoader 로 배치
    loader = DataLoader(ds, batch_size=32, shuffle=True)
    images, labels = next(iter(loader))
    print("배치:", images.shape, labels.shape, "| labels[:10] =", labels[:10].tolist())

    # ---- 클래스별 개수 세기 (3장의 집계 그대로)
    counts = {}
    for _, label in ds.items:
        counts[ds.classes[label]] = counts.get(ds.classes[label], 0) + 1
    print("클래스별 개수:", counts)

### 확인

In [ ]:
!python p02_custom_dataset.py

### 원래대로 되돌리기

파일을 책의 원래 코드로 되돌리고 싶으면 아래 셀을 실행하세요. (Colab 에서 받은 저장소 기준입니다.)

In [ ]:
!git checkout -- p02_custom_dataset.py

## 그다음

14장 패키지의 `data/` 에 `shapes.py` 로 넣고 `--dataset shapes` 로 학습되게 등록하세요. 이 과제는 14장 패키지(`code/14`)를 고치는 일입니다. 왼쪽 폴더 아이콘에서 `code/14` 의 파일을 열어 고치거나, 셀 첫 줄에 `%%writefile ../14/networks/새파일.py` 처럼 적어 저장한 뒤 아래 셀로 확인하세요.

In [ ]:
# 여기에 이어서 작성하세요